# 🚀 Сборка APK - Магический Артефакт

## Инструкция:
1. **СНАЧАЛА** загрузите ZIP проекта (Files → Upload file)
2. **ЗАТЕМ** запустите ячейки по очереди (Shift+Enter)

Ячейка 1: Распаковка проекта  
Ячейка 2: Установка зависимостей  
Ячейка 3: Android SDK  
Ячейка 4: **СБОРКА APK** (60-120 минут в первый раз)  
Ячейка 5: Скачивание результата


## ☝️ ПЕРЕД НАЧАЛОМ:

1. На своем ПК сожмите папку `opencode` в ZIP
2. В Colab нажмите `Files` (слева) → `Upload file`
3. Выберите `opencode.zip`
4. Дождитесь загрузки
5. **ПОТОМ** запустите первую ячейку ниже

In [ ]:
# ============================================
# ЯЧЕЙКА 1: РАСПАКОВКА ПРОЕКТА
# ============================================

import zipfile
import os
import glob

print("Поиск ZIP файла...")

# Ищем ZIP в /content
zip_files = glob.glob('/content/*.zip')

if not zip_files:
    print("❌ ОШИБКА: ZIP файл не найден!")
    print("")
    print("Действия:")
    print("1. На своем ПК: сожмите папку opencode в ZIP")
    print("2. В Colab: Files (слева) → Upload file")
    print("3. Выберите opencode.zip")
    print("4. Дождитесь загрузки")
    print("5. Запустите эту ячейку еще раз (Shift+Enter)")
else:
    zip_path = zip_files[0]
    print(f"✓ Найден ZIP: {zip_path}")
    print("Распаковка проекта...")
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')
    
    print("✓ Проект распакован")
    
    # Ищем папку opencode
    opencode_path = None
    if os.path.exists('/content/opencode'):
        opencode_path = '/content/opencode'
    elif os.path.exists('/content/magic-artifact'):
        opencode_path = '/content/magic-artifact'
    else:
        # Ищем первую папку
        for item in os.listdir('/content'):
            if os.path.isdir(f'/content/{item}') and item != '__pycache__':
                opencode_path = f'/content/{item}'
                break
    
    if opencode_path:
        os.chdir(opencode_path)
        print(f"✓ Переходим в: {opencode_path}")
        print()
        print("Файлы проекта:")
        files = os.listdir('.')
        for f in sorted(files)[:15]:
            print(f"  • {f}")
        print()
        print("✓ ГОТОВО К СБОРКЕ")
    else:
        print("❌ ОШИБКА: Папка проекта не найдена!")

In [ ]:
# ============================================
# ЯЧЕЙКА 2: УСТАНОВКА ЗАВИСИМОСТЕЙ
# ============================================

print("Обновление системы...")
!apt-get update > /dev/null 2>&1

print("Установка обязательных сборочных утилит...")
!apt-get install -y build-essential libtool autoconf automake > /dev/null 2>&1

print("Установка Java...")
!apt-get install -y openjdk-11-jdk-headless > /dev/null 2>&1

print("Установка Python инструментов...")
!pip install --upgrade pip setuptools wheel > /dev/null 2>&1
!pip install buildozer cython > /dev/null 2>&1

print()
print("✓ Build tools установлены")
print()
print("✓ Java установлена")
!java -version 2>&1 | grep version
print()
print("✓ buildozer установлен")
!buildozer --version
print()
print("✓ Все зависимости установлены!")

In [ ]:
# ============================================
# ЯЧЕЙКА 3: УСТАНОВКА ANDROID SDK
# ============================================
# ВНИМАНИЕ: Это может занять 10-15 минут!

import os
import subprocess

print("⏳ Установка Android SDK...")
print("Это займет 10-15 минут, не закрывайте браузер!")
print()

sdk_path = os.path.expanduser("~/android-sdk")
os.makedirs(sdk_path, exist_ok=True)

os.chdir(sdk_path)

print("Скачивание Android SDK Command-line Tools...")
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
print("✓ Скачан")

print("Распаковка...")
!unzip -o -q commandlinetools-linux-*.zip
!rm commandlinetools-linux-*.zip
print("✓ Распакован")

print()
print("Установка компонентов SDK (платформа, build-tools, NDK)...")
!yes | ./cmdline-tools/bin/sdkmanager --install "platforms;android-31" "build-tools;36.0.0" "ndk;25.1.8937393" 2>/dev/null || true
print()
print("✓ Android SDK установлен!")
print()
print("Информация:")
print(f"  SDK путь: {sdk_path}")
print(f"  Android API: 31")
print(f"  Build-tools: 36.0.0")
print(f"  NDK версия: 25.1.8937393")

In [ ]:
# ============================================
# ЯЧЕЙКА 4: СБОРКА APK (С ПОДРОБНЫМ ЛОГГИРОВАНИЕМ)
# ============================================
# ВНИМАНИЕ: Это может занять 60-120 минут в первый раз!

import os
import subprocess
import time
import glob

print("="*70)
print("🔨 НАЧАЛО СБОРКИ APK С ПОДРОБНЫМ ЛОГГИРОВАНИЕМ")
print("="*70)
print()
print("⏳ Это займет 60-120 минут в первый раз...")
print()

# Устанавливаем переменные окружения
os.environ["ANDROID_SDK_ROOT"] = os.path.expanduser("~/android-sdk")
os.environ["ANDROID_NDK_ROOT"] = os.path.expanduser("~/android-sdk/ndk/25.1.8937393")
os.environ["ANDROID_HOME"] = os.path.expanduser("~/android-sdk")

# ВАЖНО: Находим папку проекта
project_dir = None
possible_paths = ['/content/opencode', '/content/magic-artifact', '/workspace']

for item in os.listdir('/content'):
    path = f'/content/{item}'
    if os.path.isdir(path) and os.path.exists(f'{path}/buildozer.spec'):
        possible_paths.insert(0, path)

for path in possible_paths:
    if os.path.exists(path) and os.path.exists(os.path.join(path, 'buildozer.spec')):
        project_dir = path
        break

if not project_dir:
    print("❌ ОШИБКА: buildozer.spec не найден!")
else:
    print(f"✓ Папка проекта: {project_dir}")
    print(f"✓ SDK: {os.environ['ANDROID_SDK_ROOT']}")
    print(f"✓ NDK: {os.environ['ANDROID_NDK_ROOT']}")
    print()
    
    os.chdir(project_dir)
    
    # Очищаем старые логи
    if os.path.exists('build.log'):
        os.remove('build.log')
    
    print("="*70)
    print("ЗАПУСК BUILDOZER...")
    print("="*70)
    print()
    
    start_time = time.time()
    
    # Запускаем buildozer с логгированием
    process = subprocess.Popen(
        ["bash", "-c", "yes | buildozer android debug 2>&1"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=project_dir
    )
    
    # Читаем вывод строка за строкой
    line_count = 0
    for line in process.stdout:
        line_count += 1
        
        # Выводим каждую 50-ю строку для отслеживания прогресса
        if line_count % 50 == 0:
            elapsed = time.time() - start_time
            print(f"[{line_count:5d} строк, {elapsed/60:.1f} мин] {line.rstrip()[:70]}")
        
        # Выводим важные события
        if any(x in line for x in ['ERROR', 'FAILED', 'Successfully', 'created', 'APK', 'error:']):
            print(f"[ВАЖНО] {line.rstrip()}")
    
    returncode = process.wait()
    elapsed = time.time() - start_time
    
    print()
    print("="*70)
    print(f"Buildozer завершен. Код ошибки: {returncode} (время: {elapsed/60:.1f} мин)")
    print("="*70)
    print()
    
    # ВАЖНО: Проверяем физически наличие APK файла
    print("ПРОВЕРКА НАЛИЧИЯ APK ФАЙЛА...")
    print("-"*70)
    print()
    
    apk_search_paths = [
        os.path.expanduser('~/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a/dist'),
        os.path.expanduser('~/.buildozer/android/platform/build-armv7a_arm64-v8a/dist'),
        os.path.expanduser('~/.buildozer/android/platform/build/dist'),
        os.path.join(project_dir, 'bin'),
    ]
    
    found_apk = False
    apk_file = None
    
    for search_path in apk_search_paths:
        if os.path.exists(search_path):
            print(f"Ищем в: {search_path}")
            apk_files = glob.glob(os.path.join(search_path, '*.apk'))
            if apk_files:
                found_apk = True
                apk_file = max(apk_files, key=os.path.getmtime)
                size = os.path.getsize(apk_file) / (1024*1024)
                print(f"  ✅ НАЙДЕН: {os.path.basename(apk_file)}")
                print(f"  Размер: {size:.1f} MB")
                print()
                break
    
    if found_apk and apk_file:
        print()
        print("="*70)
        print("✅ APK УСПЕШНО СОБРАН!")
        print("="*70)
        print()
        print(f"Файл: {apk_file}")
        print()
        print("Следующий шаг: запустите ячейку 5 (скачивание APK)")
    else:
        print()
        print("="*70)
        print("❌ APK НЕ НАЙДЕН")
        print("="*70)
        print()
        print(f"Buildozer вернул код: {returncode}")
        print()
        print("⚠️ АНАЛИЗ ПРОБЛЕМЫ:")
        print()
        print("Проверьте последние строки build.log на ошибки:")
        print("-"*70)
        
        if os.path.exists('build.log'):
            with open('build.log', 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.readlines()
            print(f"Всего строк в логе: {len(lines)}")
            print()
            print("Последние 100 строк:")
            for line in lines[-100:]:
                print(line.rstrip())

In [ ]:
# ============================================
# ЯЧЕЙКА 5: СКАЧИВАНИЕ APK
# ============================================

import os
import glob
import shutil
import subprocess

print("="*60)
print("📥 СКАЧИВАНИЕ APK")
print("="*60)
print()

# Ищем APK рекурсивно
result = subprocess.run(
    ["find", "/root/.buildozer", "-name", "*.apk", "-type", "f"],
    capture_output=True,
    text=True,
    timeout=30
)

apk_files = [f for f in result.stdout.strip().split('\\n') if f]

if apk_files:
    # Берем самый свежий
    apk_file = max(apk_files, key=os.path.getmtime)
    
    file_size = os.path.getsize(apk_file) / (1024*1024)
    file_name = os.path.basename(apk_file)
    
    print(f"✓ APK найден!")
    print()
    print(f"Файл:   {file_name}")
    print(f"Размер: {file_size:.1f} MB")
    print(f"Путь:   {apk_file}")
    print()
    
    # Копируем в /content
    dest = f'/content/{file_name}'
    if apk_file != dest:
        print("Копирование в /content для скачивания...")
        shutil.copy(apk_file, dest)
        print("✓ Скопирован")
    
    print()
    print("-"*60)
    print()
    print("📥 КАК СКАЧАТЬ:")
    print()
    print("1. Слева нажмите значок Files (📁)")
    print()
    print(f"2. Найдите файл в корне: {file_name}")
    print()
    print("3. Щелкните правой кнопкой → Download")
    print()
    print("-"*60)
    print()
    print("✅ ГОТОВО!")
    
else:
    print("❌ APK файлы не найдены")
    print()
    print("Проверьте что ячейка 4 выполнилась успешно")